In [1]:
import re
import string
from collections import Counter, OrderedDict
import nltk
nltk.download('universal_tagset')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package universal_tagset to
[nltk_data]     C:\Users\Alex\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\universal_tagset.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\Alex\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger.zip.


True

In [2]:
def clear_punctuation(input_string):
    '''
    Clear all puncuation in a string.
    '''
    regex = re.compile('[%s]' % re.escape(string.punctuation))
    return regex.sub('', input_string)

def list_and_count(string):
    '''
    Splits a string into a set of words that excludes common words
    Counts original word appearances in a dictionary
    '''

    common_words = ['', 'the', 'of', 'to', 'and', 'a', 'in', 'is', 'it', 'you', 'that', 'he', 'was', 'for', 'on', 'are', 'with', 'as', 'i', 'his', 'they', 'be', 'at', 'one', 'have', 'this', 'from', 'or', 'had', 'by', 'not', 'word', 'but', 'what', 'some', 'we', 'can', 'out', 'other', 'were', 'all', 'there', 'when', 'up', 'use', 'your', 'how', 'said', 'an', 'each', 'she', 'which', 'do', 'their', 'time', 'if', 'will', 'way', 'about', 'many', 'then', 'them', 'write', 'would', 'like', 'so', 'these', 'her', 'long', 'make', 'thing', 'see', 'him', 'two', 'has', 'look', 'more', 'day', 'could', 'go', 'come', 'did', 'number', 'sound', 'no', 'most', 'people', 'my', 'over', 'know', 'water', 'than', 'call', 'first', 'who', 'may', 'down', 'side', 'been', 'now', 'find', 'any', 'new', 'work', 'part', 'take', 'get', 'place', 'made', 'live', 'where', 'after', 'back', 'little', 'only', 'round', 'man', 'year', 'came', 'show', 'every', 'good', 'me', 'give', 'our', 'under', 'name', 'very', 'through', 'just', 'form', 'sentence', 'great', 'think', 'say', 'low', 'line', 'differ', 'turn', 'cause', 'much', 'mean', 'before', 'move', 'right', 'boy', 'old', 'too', 'same', 'tell', 'does', 'set', 'three', 'want', 'air', 'well', 'also', 'small', 'end', 'put', 'home', 'read', 'hand', 'port', 'large', 'spell', 'add', 'even', 'land', 'here', 'must', 'big', 'high', 'such', 'follow', 'act', 'why', 'ask', 'men', 'change', 'went', 'light', 'kind', 'off', 'need', 'house', 'picture', 'try', 'us', 'again', 'animal', 'point', 'mother', 'world', 'near', 'self', 'earth', 'father', 'head', 'stand', 'own', 'page', 'should', 'country', 'found', 'answer', 'school', 'still', 's', 'www', 'll']

    words = clear_punctuation(string.lower())
    words = re.findall(r'\b[a-zA-Z0-9]+\b', words)
    distilled_words = []
    for word in words:
        if word not in common_words:
            distilled_words.append(word)

    counted_words = Counter(distilled_words)
    set_words = set(distilled_words)

    return set_words, counted_words

def pos_tag(words):
    '''
    Returns a dictionary where words are organized by their part of speech.
    '''
    pos_tags = nltk.pos_tag(words, tagset='universal')
    
    pos_dict = {}
    for word, pos in pos_tags:
        if pos in pos_dict:
            pos_dict[pos].append(word)
        else:
            pos_dict[pos] = [word]
    
    return pos_dict

def summary(freqWord_dict):
    number_words = 0
    total_value = 0
    for k,v in freqWord_dict.items():
        number_words += len(v)
        total_value += len(v)*int(k)

    length_adjusted_value = total_value/number_words

    return number_words, total_value, length_adjusted_value


In [3]:
with open("CV.txt", "r", encoding="utf-8") as text:
    cv = text.read()
    cv_set, cv_dict = list_and_count(cv)

with open("JobDescription.txt", "r", encoding="utf-8") as text:
    job = text.read()
    job_set, job_dict = list_and_count(job)

used_set = cv_set & job_set
missing_set = job_set - cv_set
useless_set = cv_set - job_set

In [4]:
print("\n\n-------------------------------------------USELESS WORDS-------------------------------------------\n")
# Not divided in POS
useless_dict = {}
for word in useless_set:
    if cv_dict[word] in useless_dict: # If word frequency in useless dict
        useless_dict[cv_dict[word]].append(word)
    else:
        useless_dict[cv_dict[word]] = [word]
n,tv,av = summary(useless_dict)
print(f"NUMBER: {n}, VALUE: {tv}, SIZE ADJ VALUE: {av}")

# Divided in POS
useless_pos_dict = pos_tag(useless_set)
for pos, list in useless_pos_dict.items():
    temp_dict_per_pos = {}
    for word in list:
        if cv_dict[word] in temp_dict_per_pos: # If word frequency in useless dict
            temp_dict_per_pos[cv_dict[word]].append(word)
        else:
            temp_dict_per_pos[cv_dict[word]] = [word]
    useless_pos_dict[pos] = temp_dict_per_pos

for pos, dict in useless_pos_dict.items():
    print(f"\nuseless {pos}:")
    for value_list_tuple in sorted(dict.items(), reverse=True):
        print(value_list_tuple)

print("\n\n-------------------------------------------MISSING WORDS-------------------------------------------\n")
# Not divided in POS
missing_dict = {}
for word in missing_set:
    if job_dict[word] in missing_dict:
        missing_dict[job_dict[word]].append(word)
    else:
        missing_dict[job_dict[word]] = [word]
n,tv,av = summary(missing_dict)
print(f"NUMBER: {n}, VALUE: {tv}, SIZE ADJ VALUE: {av}")

# Divided in POS
missing_pos_dict = pos_tag(missing_set)
for pos, list in missing_pos_dict.items():
    temp_dict_per_pos = {}
    for word in list:
        if job_dict[word] in temp_dict_per_pos: # If word frequency in missing dict
            temp_dict_per_pos[job_dict[word]].append(word)
        else:
            temp_dict_per_pos[job_dict[word]] = [word]
    missing_pos_dict[pos] = temp_dict_per_pos

for pos, dict in missing_pos_dict.items():
    print(f"\nmissing {pos}:")
    for value_list_tuple in sorted(dict.items(), reverse=True):
        print(value_list_tuple)

print("\n\n---------------------------------------------USED WORDS--------------------------------------------\n")
# Not divided in POS
used_dict = {}
for word in used_set:
    if job_dict[word] in used_dict:
        used_dict[job_dict[word]].append(word)
    else:
        used_dict[job_dict[word]] = [word]
n,tv,av = summary(used_dict)
print(f"NUMBER: {n}, VALUE: {tv}, SIZE ADJ VALUE: {av}")

# Divided in POS
used_pos_dict = pos_tag(used_set)
for pos, list in used_pos_dict.items():
    temp_dict_per_pos = {}
    for word in list:
        if job_dict[word] in temp_dict_per_pos: # If word frequency in used dict
            temp_dict_per_pos[job_dict[word]].append(word)
        else:
            temp_dict_per_pos[job_dict[word]] = [word]
    used_pos_dict[pos] = temp_dict_per_pos

for pos, dict in used_pos_dict.items():
    print(f"\nused {pos}:")
    for value_list_tuple in sorted(dict.items(), reverse=True):
        print(value_list_tuple)



-------------------------------------------USELESS WORDS-------------------------------------------

NUMBER: 169, VALUE: 193, SIZE ADJ VALUE: 1.1420118343195267

useless ADJ:
(2, ['inferential', 'card', 'webflow'])
(1, ['uncovered', 'axiomatic', 'february', 'implemented', 'identity', 'numpy', 'protocol', 'german', 'january', 'lean', 'persona', 'secondary', 'usercentered', 'whatsapp', 'daily', 'berlin', 'international', 'guidelinesvisual', 'american', 'ga4', 'forgotten', 'aloud', 'python', 'descriptive', 'intern', 'oman'])

useless VERB:
(3, ['designed'])
(2, ['mapping', 'ab', 'sorting', 'muscat'])
(1, ['sitemapping', 'producing', 'benchmarking', 'shared', 'identified', 'tracking', 'camping', 'ensured', 'email', 'researching', 'adopted', 'expressed', 'assisted', 'converted', 'reduced', 'gathering', 'alejandro', 'upwork', 'features', 'engaging', 'preprocessing', 'scrums', 'streamlined', 'aigenerated', 'specializing', 'inviting', 'crossfit', 'performing', 'led', 'highlander', 'a2', 'clo

In [5]:
string = 'aabbc'
encoding = ''
index = 0
count = 1
while index < len(string):
    if index + 1 < len(string) and string[index] == string[index + 1]:
        count += 1
    else:
        encoding += str(count) + string[index]
        count = 1
    index += 1
print(encoding)  # Output: '2a2b1c'


2a2b1c
